# GDrive Transfer with Rclone and Google Colab
This notebook sets up rclone, clones the repository, and runs the Web-UI to transfer files between cloud drives.

In [ ]:
# @title Install Dependencies & Setup Rclone
import os
import subprocess
import sys
import shutil

# 1. Navigate to the correct directory or clone the repository
if not os.path.exists("src/main.py"):
    if os.path.exists("GDrive-transfer/src/main.py"):
        os.chdir("GDrive-transfer")
        print(f"Moved to project directory: {os.getcwd()}")
    else:
        print("Cloning repository...")
        result = subprocess.run("git clone https://github.com/Hikaner/GDrive-transfer.git", shell=True, capture_output=True, text=True)
        if result.returncode == 0 and os.path.exists("GDrive-transfer"):
            os.chdir("GDrive-transfer")
            print(f"Cloned and moved to: {os.getcwd()}")
        else:
            print("Error cloning repository:")
            print(result.stderr)
            sys.exit("Failed to setup project directory.")
else:
    print(f"Already in project directory: {os.getcwd()}")

# 2. Install Rclone if not installed
if not shutil.which("rclone"):
    print("Installing rclone...")
    subprocess.run("curl https://rclone.org/install.sh | bash", shell=True)
else:
    print("rclone is already installed.")

# 3. Install Cloudflared if not installed
if not shutil.which("cloudflared"):
    print("Installing cloudflared...")
    subprocess.run("curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared.deb && rm cloudflared.deb", shell=True)
else:
    print("cloudflared is already installed.")

# 4. Install Python dependencies
print("Installing python dependencies...")
subprocess.run("pip install -q fastapi uvicorn python-multipart", shell=True)

print("\nSetup completed successfully!")

In [ ]:
# @title Run Web-UI & Expose with Cloudflared
import os
import subprocess
import time
import secrets
import threading
import sys
import re

# 1. Ensure we are in the correct directory
if not os.path.exists("src/main.py"):
    if os.path.exists("GDrive-transfer/src/main.py"):
        os.chdir("GDrive-transfer")
    else:
        sys.exit("Error: src/main.py not found. Please run Cell 2 first.")

# 2. Clean up any existing uvicorn or cloudflared processes from previous runs
print("Cleaning up existing processes...")
if sys.platform != "win32":
    subprocess.run("pkill -f uvicorn", shell=True)
    subprocess.run("pkill -f cloudflared", shell=True)
time.sleep(1)

# 3. Generate a random token for authentication
AUTH_TOKEN = secrets.token_hex(16)
os.environ["AUTH_TOKEN"] = AUTH_TOKEN

# 4. Start FastAPI server in the background
print("Starting FastAPI server...")
backend_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"],
    env=os.environ.copy(),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait for FastAPI to start
time.sleep(2)

# 5. Start cloudflared tunnel in the background
print("Starting cloudflared tunnel...")
cloudflared_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 6. Monitor cloudflared output to extract the public URL
def monitor_tunnel():
    for line in iter(cloudflared_process.stdout.readline, ''):
        if "trycloudflare.com" in line and "dest=" not in line and "Request" not in line:
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if match:
                url = match.group(0)
                print("\n" + "="*80)
                print(f" GDrive-transfer Web-UI is live!")
                print(f" Access URL: {url}/?token={AUTH_TOKEN}")
                print("="*80 + "\n")
                break

threading.Thread(target=monitor_tunnel, daemon=True).start()

# 7. Keep the cell running and monitor process health
try:
    while True:
        if backend_process.poll() is not None:
            print("FastAPI server stopped unexpectedly.")
            break
        if cloudflared_process.poll() is not None:
            print("Cloudflared tunnel stopped unexpectedly.")
            break
        time.sleep(1)
except KeyboardInterrupt:
    print("\nStopping services...")
    backend_process.terminate()
    cloudflared_process.terminate()
    backend_process.wait()
    cloudflared_process.wait()
    print("Services stopped successfully.")